In [ ]:
CSV_FILE = "/data/philei/tt-metal/generated/profiler/reports/2026_02_24_15_34_58/ops_perf_results_2026_02_24_15_34_58.csv"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact


In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv(CSV_FILE)

# Rename the columns for clarity
df = df.rename(columns={
    'OP CODE': 'Operation',
    'HOST DURATION [ns]': 'Host Time',
    'OP TO OP LATENCY [ns]': 'Time Between Ops',
    'DEVICE FW DURATION [ns]': 'Device Time'
})


# Filter out rows before compilation finished
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('compilation_finished', na=False)
matching_indices = df.index[mask]
assert not matching_indices.empty, "No 'compilation_finished' found in ProfilerNoopOperation attributes"
latest_compilation_flag = matching_indices[-1]
df = df.iloc[latest_compilation_flag + 1:]

# Find number of training steps
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('iteration_', na=False)
matching_indices = df.index[mask]
num_training_steps = 3 # for backward compatibility, will be removed before merge
if not matching_indices.empty:
    filtered_df = df[df.index.isin(matching_indices)]
    num_training_steps = len(filtered_df['ATTRIBUTES'].unique())

df = df[df['Operation'] != 'ProfilerNoopOperation']

all_operations = df['Operation'].unique()

num_training_steps

In [ ]:
def draw_diagrams_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']
    topk = 15

    for col in time_columns:
        # ----- top-k + “Others” slice -------------------------------------
        sorted_times = grouped[col].sort_values(ascending=False)
        top_times    = sorted_times.head(topk)
        others_sum   = sorted_times.iloc[topk:].sum()
        if others_sum:
            top_times = pd.concat([top_times, pd.Series({'Others': others_sum})])

        labels = top_times.index.tolist()
        sizes  = top_times.values
        pct    = 100 * sizes / sizes.sum()

        # ----- plot -------------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 8))
        wedges, _ = ax.pie(sizes, startangle=140)   # no autopct ⇒ nothing on pie

        legend_text = [f'{lbl} — {p:.1f}%' for lbl, p in zip(labels, pct)]
        ax.legend(
            wedges,
            legend_text,
            title=f'Operations (share of {aggregation})',
            loc='center left',
            bbox_to_anchor=(1, 0.5)
        )

        ax.set_title(f'Top {topk} Operations by {aggregation} {col}')
        ax.axis('equal')
        plt.show()

        # display(fig)    # ➋ show it once

In [ ]:
draw_diagrams_with_aggregation('sum')

In [ ]:
draw_diagrams_with_aggregation('mean')

In [ ]:
def name_per_aggregation(aggregation):
    if aggregation == 'sum':
        return 'Total (per training step)'
    elif aggregation == 'mean':
        return 'Average'
    else:
        raise ValueError(f"Unsupported aggregation: {aggregation}")
    
def draw_charts_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']

    # Loop through each time column to create horizontal bar charts and pie charts
    for col in time_columns:
        # Extract the total times per operation for the current column
        total_times = grouped[col] / 1_000_000  # Convert from nanoseconds to milliseconds
        if aggregation == 'sum':
            total_times /= num_training_steps  # Normalize by number of training steps if aggregation is 'sum'
        
        # Create a horizontal bar chart
        plt.figure(figsize=(10, 6))
        total_times.sort_values().plot(kind='barh', color='skyblue') 
        plt.title(f'{name_per_aggregation(aggregation)} {col} per Operation')
        plt.xlabel(f'{name_per_aggregation(aggregation)} {col} (ms)')
        plt.ylabel('Operation')
        plt.tight_layout()
        plt.show()

In [ ]:
draw_charts_with_aggregation('sum')

In [ ]:
draw_charts_with_aggregation('mean')

In [ ]:
@interact(operation=all_operations)
def draw_per_operation_stats(operation):
    df_op = df[df['Operation'] == operation]
    if df_op.empty:
        print(f"No data available for operation: {operation}")
        return

    metrics = [
        ('Host Time (ms)',          df_op['Host Time']        / 1_000_000),
        ('Time Between Ops (ms)',   df_op['Time Between Ops'] / 1_000_000),
        ('Device Time (ms)',        df_op['Device Time']      / 1_000_000),
    ]

    for title, series in metrics:
        # --- build figure explicitly so we know which one to close ------------
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(series.index, series.values, marker='o', color='red')
        ax.set_title(f'{operation} – {title}')
        ax.set_xlabel('Index')
        ax.set_ylabel(title)
        ax.grid(True)
        fig.tight_layout()
        plt.show()

In [ ]:
# example how to manually extract performance data for a specific operation
e = df[df['Operation'] == 'Untilize']
e = e[['Operation', 'Host Time', 'Time Between Ops', 'Device Time']]
e

In [ ]:
metrics = ['Host Time', 'Time Between Ops', 'Device Time']

def anomaly_detection_per_operation(operation, metric):
    df_op = df[df['Operation'] == operation]
    if df_op.empty:
        print(f"No data available for operation: {operation}")
        return

    # Calculate the mean and standard deviation for each metric
    
    series = df_op[metric]
    
    mean = series.mean()
    std_dev = series.std()
    
    # Identify anomalies as points that are more than 3 standard deviations from the mean
    anomalies = series[(series < mean - 3 * std_dev) | (series > mean + 3 * std_dev)]
    
    if not anomalies.empty:
        return df_op.loc[anomalies.index]
    return None

In [ ]:
import json

def is_not_nan(value):
    """Check if a value is not NaN."""
    return pd.notna(value) and value != 'NaN' and value != 'nan' and value != ''

@interact(operation=all_operations, metric_name=metrics)
def show_anomalies_attributes_per_metric(operation, metric_name):
    anomaly_df = anomaly_detection_per_operation(operation, metric_name)

    if anomaly_df is None:
        print(f"No anomalies detected for {operation} in {metric_name}.")
        return

    for index, row in anomaly_df.iterrows():
        attr = row['ATTRIBUTES']
        core_count = row['CORE COUNT']
        metric_value = row[metric_name]

        # find all columns with prefix `INPUT_`
        input_columns = [col for col in row.index if col.startswith('INPUT_')]
        input_values = {col: row[col] for col in input_columns if is_not_nan(row[col])}

        # improve print of dictionary with json
        input_values = json.dumps(input_values, indent=8)

        if isinstance(attr, str):
            attr = attr.replace(';', ',')
            attr = attr.replace('\'', '"')
            try:
                attr = json.loads(attr)
            except:
                pass
            attr = json.dumps(attr, indent=8) if isinstance(attr, dict) else attr

        print(f"Anomaly at index {index}: ")
        print(f"    {metric_name} = {metric_value / 1_000_000} ms")
        print(f"    core count = {core_count}")
        print(f"    inputs = {input_values}")
        print(f"    attributes = {attr}")


In [ ]:
# ── Training Phase Timing Analysis from Profiler Markers ────────────────
# Uses extract_results.py for parsing, then visualizes.

import sys
from pathlib import Path

TT_TRAIN = Path("/data/philei/tt-metal/tt-train")
if str(TT_TRAIN / "tools") not in sys.path:
    sys.path.insert(0, str(TT_TRAIN / "tools"))

from profiling.results_json.extract_results import extract_timings, print_timings, PHASE_COLS

timings = extract_timings(Path(CSV_FILE))
assert timings is not None, "No timing data found in CSV"

# Print per-device tables
print_timings(timings)

# ── Visualize (first device) ──
first_dev_key = next(iter(timings))
dev_data = timings[first_dev_key]
iters = dev_data["iterations"]
avail_phases = [c for c in PHASE_COLS if any(it.get(c, 0) > 0 for it in iters)]
phase_labels = [c.replace("_ms", "").replace("_", " ").title() for c in avail_phases]

iters_df = pd.DataFrame(iters)
first_iter = iters_df["iteration"].min()
steady = iters_df[iters_df["iteration"] > first_iter]
avg = steady[avail_phases].mean()

if not steady.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0']

    ax = axes[0]
    avg.plot(kind='bar', ax=ax, color=colors[:len(avail_phases)])
    ax.set_xticklabels(phase_labels, rotation=30, ha='right')
    ax.set_title(f'{first_dev_key}: Average Phase Duration (excl. iter {first_iter})')
    ax.set_ylabel('Time (ms)')
    for i_bar, v in enumerate(avg):
        ax.text(i_bar, v + avg.max() * 0.01, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

    ax = axes[1]
    for ci, col in enumerate(avail_phases):
        ax.plot(steady['iteration'], steady[col], marker='o',
                label=phase_labels[ci], color=colors[ci % len(colors)])
    ax.set_title(f'{first_dev_key}: Phase Duration per Iteration (excl. iter {first_iter})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Time (ms)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Op-by-op execution timeline for one iteration ──
# Shows each op in execution order (not grouped), so you can see
# the actual sequence and spot slow individual ops.

TIMELINE_DEVICE_ID = 0       # which device to plot
TIMELINE_ITERATION = 3       # which iteration (post-compilation)
X_TICK_INTERVAL_MS = 0.5     # x-axis tick spacing in ms

raw_tl = pd.read_csv(CSV_FILE)

# Strip everything up to compilation_finished
comp_mask = (raw_tl['OP CODE'] == 'ProfilerNoopOperation') & \
            raw_tl['ATTRIBUTES'].str.contains('compilation_finished', na=False)
raw_tl = raw_tl.iloc[raw_tl.index[comp_mask][-1] + 1:].reset_index(drop=True)

# Find iteration boundaries from markers
import re as _re
raw_tl['_ident'] = None
noop = raw_tl['OP CODE'] == 'ProfilerNoopOperation'
raw_tl.loc[noop, '_ident'] = raw_tl.loc[noop, 'ATTRIBUTES'].str.extract(
    r"'identifier':\s*'([^']+)'", expand=False
)

iter_start_marker = f'iteration_{TIMELINE_ITERATION}'
iter_end_marker = f'iteration_{TIMELINE_ITERATION + 1}'

start_idx = raw_tl.index[raw_tl['_ident'] == iter_start_marker]
end_idx = raw_tl.index[raw_tl['_ident'] == iter_end_marker]

if start_idx.empty:
    print(f"Iteration {TIMELINE_ITERATION} not found")
else:
    s = start_idx[0]
    e = end_idx[0] if not end_idx.empty else len(raw_tl)
    
    # Filter to one device, one iteration, non-marker ops
    chunk = raw_tl.iloc[s:e].copy()
    chunk = chunk[(chunk['DEVICE ID'] == TIMELINE_DEVICE_ID) & (chunk['_ident'].isna())]
    chunk = chunk.reset_index(drop=True)
    
    if chunk.empty:
        print(f"No ops for device {TIMELINE_DEVICE_ID} in iteration {TIMELINE_ITERATION}")
    else:
        CLOCK_GHZ = 1.35
        chunk['duration_ms'] = (
            chunk['DEVICE FW END CYCLE'] - chunk['DEVICE FW START CYCLE']
        ) / (CLOCK_GHZ * 1e6)
        
        fig, ax = plt.subplots(figsize=(14, max(6, len(chunk) * 0.12)))
        
        y_pos = range(len(chunk))
        colors = ['#F44336' if d > chunk['duration_ms'].quantile(0.95) else '#2196F3' 
                  for d in chunk['duration_ms']]
        
        ax.barh(y_pos, chunk['duration_ms'], color=colors, height=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(chunk['OP CODE'], fontsize=6)
        ax.invert_yaxis()
        import numpy as _np
        x_max = chunk['duration_ms'].max()
        ax.set_xticks(_np.arange(0, x_max + X_TICK_INTERVAL_MS, X_TICK_INTERVAL_MS))
        ax.set_xlabel('Duration (ms)')
        ax.set_title(
            f'Device {TIMELINE_DEVICE_ID}, Iteration {TIMELINE_ITERATION}: '
            f'Op Execution Timeline ({len(chunk)} ops, '
            f'total={chunk["duration_ms"].sum():.1f} ms)'
        )
        ax.grid(True, axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nTop 10 slowest ops:")
        top = chunk.nlargest(10, 'duration_ms')[['OP CODE', 'duration_ms']].reset_index(drop=True)
        top.index += 1
        display(top)